In [1]:
# ==========================================
# PARAMETER TO BE TESTED
# ==========================================

PARAMETER = "iddq_uA"

# Other options:
#PARAMETER = "iddq_uA"
# PARAMETER = "leakage_current_nA"
# PARAMETER = "propagation_delay_ns"

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load the dataset used by Module B
df = pd.read_csv("clean_burnin_dataset.csv")

# Basic checks
print("Number of rows:", len(df))
print("Number of columns:", len(df))

print("\nColumn names and data types:")
print(df.dtypes)

# Check for completely empty columns
empty_columns = df.columns[df.isna().all()].tolist()

if len(df) > 0 and len(empty_columns) == 0:
    print("\nTEST 1 PASSED ")
    print("Dataset loaded successfully and is not empty.")
else:
    print("\nTEST 1 FAILED ")

    if len(df) == 0:
        print("Dataset contains no rows.")

    if len(empty_columns) > 0:
        print("Completely empty columns:", empty_columns)

Number of rows: 9052
Number of columns: 9052

Column names and data types:
component_id                       int64
lot_id                             int64
wafer_id                           int64
die_x_pos                          int64
die_y_pos                          int64
manufacturer_id                    int64
burn_in_chamber_id                 int64
device_family                     object
package_type                      object
pin_count                          int64
test_temperature_c               float64
supply_voltage_v                 float64
test_date                         object
iddq_uA_0h                       float64
iddq_uA_24h                      float64
iddq_uA_96h                      float64
iddq_uA_168h                     float64
leakage_current_nA_0h            float64
leakage_current_nA_24h           float64
leakage_current_nA_96h           float64
leakage_current_nA_168h          float64
propagation_delay_ns_0h          float64
propagation_delay_ns_24

In [3]:
# ==========================================
# TEST 2 — CHECK REQUIRED COLUMNS
# ==========================================

# Required columns for the selected parameter
required_columns = [
    f"{PARAMETER}_0h",
    f"{PARAMETER}_24h",
    f"{PARAMETER}_168h"
]

# Find missing columns
missing_columns = []

for column in required_columns:
    if column not in df.columns:
        missing_columns.append(column)

# Final result
if len(missing_columns) == 0:
    print("TEST 2 PASSED ")
    print(f"Parameter tested: {PARAMETER}")
    print("All required columns are present.")

else:
    print("TEST 2 FAILED ")
    print(f"Parameter tested: {PARAMETER}")
    print("Missing columns:", missing_columns)

TEST 2 PASSED 
Parameter tested: iddq_uA
All required columns are present.


In [4]:
# Count rows before cleaning
rows_before = len(df)

# Required columns for the selected parameter
required_columns = [
    f"{PARAMETER}_0h",
    f"{PARAMETER}_24h",
    f"{PARAMETER}_168h"
]

# Apply Module B's cleaning rule
expected_clean_df = df.dropna(
    subset=required_columns
).copy()

# Check that no required value is missing
missing_values = expected_clean_df[
    required_columns
].isna().sum().sum()

# Check result
if missing_values == 0:
    print("TEST 3 PASSED ")
    print(f"Parameter tested: {PARAMETER}")
    print("Rows with missing required values are removed correctly.")
    print("Rows before cleaning:", rows_before)
    print("Rows after cleaning:", len(expected_clean_df))
else:
    print("TEST 3 FAILED ")
    print(f"Parameter tested: {PARAMETER}")
    print("Missing values remaining:", missing_values)

TEST 3 PASSED 
Parameter tested: iddq_uA
Rows with missing required values are removed correctly.
Rows before cleaning: 9052
Rows after cleaning: 9052


In [5]:
# Create the cleaned dataset
clean_df_test = df.dropna(
    subset=[
        f"{PARAMETER}_0h",
        f"{PARAMETER}_24h",
        f"{PARAMETER}_168h"
    ]
).copy()


# Create the features
clean_df_test["drift_0_to_24"] = (
    clean_df_test[f"{PARAMETER}_24h"]
    - clean_df_test[f"{PARAMETER}_0h"]
)

clean_df_test["ratio_24_to_0"] = (
    clean_df_test[f"{PARAMETER}_24h"]
    / (clean_df_test[f"{PARAMETER}_0h"] + 1e-6)
)


# Select input features
X_test_check = clean_df_test[
    [
        f"{PARAMETER}_0h",
        f"{PARAMETER}_24h",
        "drift_0_to_24",
        "ratio_24_to_0"
    ]
]


# Target: 168h value
y_test_check = np.log1p(
    clean_df_test[f"{PARAMETER}_168h"]
)


# Perform the same split as Module B
X_train, X_test, y_train, y_test = train_test_split(
    X_test_check,
    y_test_check,
    test_size=0.2,
    random_state=42
)


# Expected test size
expected_test_size = int(len(clean_df_test) * 0.2)


# Check
if len(X_test) == expected_test_size:
    print("TEST 4 PASSED ")
    print(f"Parameter tested: {PARAMETER}")
    print("Train/test split uses the correct 80/20 ratio.")
    print("Training rows:", len(X_train))
    print("Testing rows:", len(X_test))
else:
    print("TEST 4 FAILED ")
    print(f"Parameter tested: {PARAMETER}")
    print("Expected test rows:", expected_test_size)
    print("Actual test rows:", len(X_test))

TEST 4 FAILED 
Parameter tested: iddq_uA
Expected test rows: 1810
Actual test rows: 1811


In [11]:
# Import the final Module B
%pip install scikit-optimize
import module_b

print("Module B imported successfully ✅")

Note: you may need to restart the kernel to use updated packages.
Module B imported successfully ✅


In [14]:
# ==========================================
# TEST 13 — CHECK PREDICTIONS ARE VALID
# ==========================================

# Generate predictions using Module B's actual model
y_pred_log_test = module_b.best_model.predict(
    module_b.X_test
)

# Convert predictions back to original scale
y_pred_test = np.expm1(y_pred_log_test)

# Check for invalid predictions
invalid_predictions = (
    ~np.isfinite(y_pred_test)
    | (y_pred_test < 0)
)

invalid_count = invalid_predictions.sum()

if invalid_count == 0:
    print("TEST 5 PASSED ")
    print(f"Parameter tested: {PARAMETER}")
    print("All predictions are valid and non-negative.")
    print("Minimum prediction:", y_pred_test.min())
    print("Maximum prediction:", y_pred_test.max())

else:
    print("TEST 5 FAILED ")
    print(f"Parameter tested: {PARAMETER}")
    print("Invalid predictions:", invalid_count)

TEST 5 PASSED 
Parameter tested: iddq_uA
All predictions are valid and non-negative.
Minimum prediction: 15.399669253233192
Maximum prediction: 806673.2079712666


In [15]:
# Calculate individual prediction errors

actual_values = np.expm1(
    module_b.y_test_log
)

individual_errors = np.abs(
    actual_values - y_pred_test
)

# Check that every error is valid
if (
    len(individual_errors) == len(actual_values)
    and np.isfinite(individual_errors).all()
    and (individual_errors >= 0).all()
):
    print("TEST 6 PASSED ")
    print("Individual prediction errors are valid.")
    print("Number of errors:", len(individual_errors))
    print("Average error:", individual_errors.mean())
else:
    print("TEST 6 FAILED ")

    if len(individual_errors) != len(actual_values):
        print("Error count does not match actual values.")

    if not np.isfinite(individual_errors).all():
        print("Invalid error values found.")

    if not (individual_errors >= 0).all():
        print("Negative error found.")

TEST 6 PASSED 
Individual prediction errors are valid.
Number of errors: 2000
Average error: 2867.5476405357954
